In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# KIT color palette
kitgreen = "#009682"
green = "#98BF64"
kitblue = "#4664AA"
grey5 = "#f2f2f2ff"
grey30 = "#b3b3b3ff"
kitred = "#A22223"
kitorange = "#DF9B1B"

# Path constants
DATA_DIR = Path("./dataset")
GEO_DIR = DATA_DIR / "geoserver"
NUTS_GPKG = DATA_DIR / "NUTS_RG_01M_2024_4326.gpkg"
GVA_XLSX = DATA_DIR / "nama_10r_3gva.xlsx"
OUTPUT_DIR = Path("./results/intermediate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Target region
BOERDE_NUTS3 = "DEE07"

# Börde Data Preparation

**Region:** Landkreis Börde (NUTS3: DEE07), Sachsen-Anhalt  
**Source region:** 38 Gemeinden (municipalities)  
**Target:** UW substations (clipped to the Börde region, including "Last aktuell in MW" load data)  
**GVA data:** Eurostat nama_10r_3gva, year 2022  

**Key differences from the UK case study:**
- A single NUTS3 region = 1 sample (the UK case has 16 ITL2 regions)
- GVA percent is a NUTS3-wide value, shared by all Gemeinden
- CRS: EPSG:25832 (UTM32N)

**Outputs:**
- `source_regions.gpkg` -- 38 Gemeinden + population + GVA percent + area share
- `substations.gpkg` -- UW + load + Gemeinde assignment

## 1. Load Gemeinden (Source Regions)

In [ ]:
# Load 38 Gemeinden (municipalities) as source regions
gemeinden_raw = gpd.read_file(GEO_DIR / "Postgres_v_lkb_vg250_gem_sql.geojson")
print(f"Raw Gemeinden: {len(gemeinden_raw)}")

# Load the Kreis boundary (used for clipping below)
kreis = gpd.read_file(GEO_DIR / "Postgres_v_lkb_vg250_krs_sql.geojson")
kreis_pop = kreis.iloc[0]['Einwohner']

# Spatial filter: keep only Gemeinden whose centroid lies within the Börde region
# The GeoServer data contains stray records (Berlin, Hannover, Leipzig, Magdeburg, etc. do not belong to Börde)
gemeinden_raw["_centroid"] = gemeinden_raw.geometry.centroid
centroids_gdf = gemeinden_raw.set_geometry("_centroid")
within_mask = centroids_gdf.within(kreis.geometry.iloc[0])
removed = gemeinden_raw.loc[~within_mask, "Name"].tolist()
gemeinden = gemeinden_raw[within_mask].drop(columns=["_centroid"]).copy()

print(f"Gemeinden after filtering: {len(gemeinden)}")
if removed:
    print(f"Removed (not in Börde): {removed}")
print(f"\nKreis population: {kreis_pop:,.0f}")

# Population overview
n_pop_valid = gemeinden['Einwohner'].notna().sum()
pop_sum = gemeinden['Einwohner'].sum()
print(f"Gemeinden with population data: {n_pop_valid}/{len(gemeinden)}")
print(f"Known population total: {pop_sum:,.0f}")

gemeinden[["Name", "Einwohner"]].sort_values("Einwohner", ascending=False)

## 2. Population Data Processing

22/38 Gemeinden are missing Einwohner data. Strategy:
- Subtract the known Gemeinden population from the Kreis total to get the remaining population
- Distribute the remaining population to Gemeinden with missing data, proportional to area
- Compute `residential_percent = Einwohner / total_Einwohner`

In [ ]:
# Project to EPSG:25832 to compute area
gem_proj = gemeinden.to_crs("EPSG:25832")
gemeinden["area_m2"] = gem_proj.geometry.area

# Fill missing population: distribute from the Kreis total, proportional to area
known_mask = gemeinden["Einwohner"].notna()
known_pop = gemeinden.loc[known_mask, "Einwohner"].sum()

# If the known population already exceeds the Kreis total, just use the known data (fill 0)
# Otherwise distribute the remaining population proportional to area
if known_pop >= kreis_pop:
    print(f"Known population ({known_pop:,.0f}) >= Kreis total population ({kreis_pop:,.0f})")
    print("Filling missing Gemeinden population with 0, using the known total as the overall population")
    gemeinden["Einwohner"] = gemeinden["Einwohner"].fillna(0)
    total_pop = gemeinden["Einwohner"].sum()
else:
    remaining_pop = kreis_pop - known_pop
    missing_area = gemeinden.loc[~known_mask, "area_m2"].sum()
    gemeinden.loc[~known_mask, "Einwohner"] = (
        gemeinden.loc[~known_mask, "area_m2"] / missing_area * remaining_pop
    )
    total_pop = kreis_pop
    print(f"Distributed the remaining population of {remaining_pop:,.0f} to {(~known_mask).sum()} Gemeinden")

# Compute residential_percent (Germany's total population ~84M, 2022)
GERMANY_POP_2022 = 84_359_000
gemeinden["population"] = gemeinden["Einwohner"]
gemeinden["residential_percent"] = gemeinden["population"] / GERMANY_POP_2022

print(f"\nTotal population: {total_pop:,.0f}")
print(f"residential_percent range: {gemeinden['residential_percent'].min():.6f} - {gemeinden['residential_percent'].max():.6f}")
print(f"residential_percent sum: {gemeinden['residential_percent'].sum():.6f}")

gemeinden[["Name", "population", "residential_percent"]].sort_values("population", ascending=False).head(10)

## 3. GVA Data Parsing (Year 2022)

Parses the Börde (DEE07) GVA data from Eurostat's `nama_10r_3gva.xlsx`.

**NACE category mapping (aligned with the UK case's SIC07):**
- `agricultural_gva` -> Sheet 2 (Agriculture)
- `industrial_gva` -> Sheet 3 (Industry) + Sheet 5 (Construction)
- `commercial_gva` -> Sheet 6 (Retail/Transport) + Sheet 9 (Finance/RE/Professional)
- `others_gva` -> Total - agricultural - industrial - commercial

**Note:** GVA data is at the NUTS3-wide level; all 38 Gemeinden share the same GVA percent.

In [ ]:
def parse_gva_sheet(xlsx_path, sheet_name, region_name="Börde", year_col_idx=15):
    """
    Parse the data for a given sheet from the Eurostat GVA Excel file.

    Excel structure:
    - Row 8: [TIME, 2015, flag, 2016, flag, ..., 2022(col15), flag, ...]
    - Row 10: Germany total
    - Row N: individual NUTS3 regions

    Returns: (region_value, germany_value)
    """
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
    
    region_val = None
    germany_val = None
    
    for _, row in df.iterrows():
        geo_label = str(row.iloc[0])
        if geo_label == "Germany":
            val = row.iloc[year_col_idx]
            germany_val = float(val) if str(val) != ':' and pd.notna(val) else None
        if region_name in geo_label:
            val = row.iloc[year_col_idx]
            region_val = float(val) if str(val) != ':' and pd.notna(val) else None
    
    return region_val, germany_val

# Parse each NACE category
gva_sheets = {
    'total':        ('Sheet 1',  'Total - all NACE'),
    'agriculture':  ('Sheet 2',  'Agriculture'),
    'industry':     ('Sheet 3',  'Industry (except construction)'),
    'construction': ('Sheet 5',  'Construction'),
    'retail':       ('Sheet 6',  'Wholesale/Retail/Transport'),
    'finance':      ('Sheet 9',  'Finance/Real Estate/Professional'),
    'public':       ('Sheet 13', 'Public/Social/Other services'),
}

gva = {}
for key, (sheet, label) in gva_sheets.items():
    region_val, germany_val = parse_gva_sheet(GVA_XLSX, sheet)
    gva[key] = {'boerde': region_val, 'germany': germany_val}
    print(f"  {label:45s}: Börde = {region_val:>10.2f} M€, Germany = {germany_val:>12.0f} M€")

# Assemble GVA categories (aligned with the UK case's SIC07 classification)
agricultural_gva = gva['agriculture']['boerde']
industrial_gva = gva['industry']['boerde'] + gva['construction']['boerde']
commercial_gva = gva['retail']['boerde'] + gva['finance']['boerde']
others_gva = gva['total']['boerde'] - agricultural_gva - industrial_gva - commercial_gva

# Germany totals (same categories)
agricultural_gva_de = gva['agriculture']['germany']
industrial_gva_de = gva['industry']['germany'] + gva['construction']['germany']
commercial_gva_de = gva['retail']['germany'] + gva['finance']['germany']
others_gva_de = gva['total']['germany'] - agricultural_gva_de - industrial_gva_de - commercial_gva_de

print(f"\n=== Börde GVA Summary (2022, million EUR) ===")
print(f"  Agricultural: {agricultural_gva:>10.2f}  (DE: {agricultural_gva_de:>12.0f})")
print(f"  Industrial:   {industrial_gva:>10.2f}  (DE: {industrial_gva_de:>12.0f})")
print(f"  Commercial:   {commercial_gva:>10.2f}  (DE: {commercial_gva_de:>12.0f})")
print(f"  Others:       {others_gva:>10.2f}  (DE: {others_gva_de:>12.0f})")
print(f"  Total:        {gva['total']['boerde']:>10.2f}  (DE: {gva['total']['germany']:>12.0f})")

In [ ]:
# Compute GVA percent (share of the Germany-wide total) - a NUTS3-wide value shared by all Gemeinden
gemeinden["agricultural_gva"] = agricultural_gva
gemeinden["industrial_gva"] = industrial_gva
gemeinden["commercial_gva"] = commercial_gva
gemeinden["others_gva"] = others_gva

gemeinden["agricultural_percent"] = agricultural_gva / agricultural_gva_de
gemeinden["industrial_percent"] = industrial_gva / industrial_gva_de
gemeinden["commercial_percent"] = commercial_gva / commercial_gva_de
gemeinden["others_percent"] = others_gva / others_gva_de

print("GVA percent (share of Germany-wide total):")
print(f"  agricultural_percent: {gemeinden['agricultural_percent'].iloc[0]:.6f}")
print(f"  industrial_percent:   {gemeinden['industrial_percent'].iloc[0]:.6f}")
print(f"  commercial_percent:   {gemeinden['commercial_percent'].iloc[0]:.6f}")
print(f"  others_percent:       {gemeinden['others_percent'].iloc[0]:.6f}")

## 4. Area Share

In [ ]:
# Area share (Gemeinde area / total Börde area)
total_area = gemeinden["area_m2"].sum()
gemeinden["area_percent"] = gemeinden["area_m2"] / total_area

print(f"Total Börde area: {total_area / 1e6:.2f} km²")
print(f"area_percent range: {gemeinden['area_percent'].min():.4f} - {gemeinden['area_percent'].max():.4f}")
print(f"area_percent sum: {gemeinden['area_percent'].sum():.6f}")

# Add the NUTS3 identifier (single region, all Gemeinden share the same NUTS3)
gemeinden["NUTS3"] = BOERDE_NUTS3

## 5. Substation Data

Loads UW (Umspannwerk) data:
- `v_epa_uw.geojson` -- 651 UW locations (entire service area)
- `v_epa_uw_last.geojson` -- 613 UW load snapshots

Spatially clipped to the Börde county boundary, joined with load data.

In [ ]:
# Load UW data
uw_all = gpd.read_file(GEO_DIR / "Postgres_v_epa_uw.geojson")
uw_last_all = gpd.read_file(GEO_DIR / "Postgres_v_epa_uw_last.geojson")
print(f"All UW locations: {len(uw_all)}")
print(f"All UW loads: {len(uw_last_all)}")

# Spatially clip uw_last_all directly (already contains load data)
uw_last_boerde = gpd.sjoin(uw_last_all, kreis[["geometry"]], how="inner", predicate="within")
uw_last_boerde = uw_last_boerde.drop(columns=["index_right"])
print(f"\nUW within Börde (with load): {len(uw_last_boerde)}")

# Also clip uw_all to check for UW that have a location but no load record
uw_pos_boerde = gpd.sjoin(uw_all, kreis[["geometry"]], how="inner", predicate="within")
uw_pos_only = uw_pos_boerde[~uw_pos_boerde["Kennzeichen"].isin(uw_last_boerde["Kennzeichen"])]
print(f"UW within Börde (location only, no load): {len(uw_pos_only)}")

# Force conversion to numeric
uw_last_boerde["Last aktuell in MW"] = pd.to_numeric(
    uw_last_boerde["Last aktuell in MW"], errors="coerce"
).fillna(0.0)

n_nonzero = (uw_last_boerde["Last aktuell in MW"] > 0).sum()
total_load = uw_last_boerde["Last aktuell in MW"].sum()

print(f"\nNon-zero loads: {n_nonzero}")
print(f"Total load: {total_load:.1f} MW")
print(f"Load range: {uw_last_boerde['Last aktuell in MW'].min():.1f} - {uw_last_boerde['Last aktuell in MW'].max():.1f} MW")

uw_last_boerde[["Kennzeichen", "Name", "Last aktuell in MW", "Esp aktuell in MW"]].sort_values(
    "Last aktuell in MW", ascending=False
)

In [ ]:
# Assign UW to Gemeinden (sjoin)
substations = uw_last_boerde[["Kennzeichen", "Name", "Last aktuell in MW", "geometry"]].copy()
substations = substations.rename(columns={"Last aktuell in MW": "p_mw"})

# Drop substations with zero load
n_before = len(substations)
substations = substations[substations["p_mw"] > 0].copy()
print(f"Dropped zero-load UW: {n_before} -> {len(substations)}")

# sjoin to the containing Gemeinde
subs_with_gem = gpd.sjoin(substations, gemeinden[["Name", "geometry"]], how="left", predicate="within")
substations["Gemeinde"] = subs_with_gem["Name_right"].values
substations["NUTS3"] = BOERDE_NUTS3

# Handle UW not contained in any Gemeinde (boundary issues)
n_unassigned = substations["Gemeinde"].isna().sum()
if n_unassigned > 0:
    print(f"[Warning] {n_unassigned} UW not within any Gemeinde, assigning by nearest neighbor")
    for idx in substations[substations["Gemeinde"].isna()].index:
        pt = substations.loc[idx, "geometry"]
        dists = gemeinden.geometry.distance(pt)
        nearest_idx = dists.idxmin()
        substations.loc[idx, "Gemeinde"] = gemeinden.loc[nearest_idx, "Name"]

print(f"\nTotal substations: {len(substations)}")
print(f"Distribution by Gemeinde:")
print(substations.groupby("Gemeinde")["p_mw"].agg(["count", "sum"]).sort_values("sum", ascending=False))

substations

## 6. Summary Statistics & Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=150)

# Left panel: Gemeinden colored by population
ax = axes[0]
gemeinden.plot(ax=ax, column="population", cmap="YlOrRd", edgecolor="black",
               linewidth=0.5, legend=True,
               legend_kwds={"label": "Einwohner", "shrink": 0.6})
ax.set_title(f"Gemeinden population distribution (n={len(gemeinden)})", fontsize=10)
ax.set_xlabel("Longitude [°E]", fontsize=8)
ax.set_ylabel("Latitude [°N]", fontsize=8)
ax.tick_params(labelsize=6)

# Middle panel: UW + load magnitude
ax = axes[1]
gemeinden.plot(ax=ax, color=grey5, edgecolor=grey30, linewidth=0.3)
subs_plot = substations[substations["p_mw"] > 0].copy()
if len(subs_plot) > 0:
    subs_plot.plot(ax=ax, column="p_mw", cmap="YlOrRd", markersize=40, legend=True,
                   legend_kwds={"label": "Last [MW]", "shrink": 0.6}, zorder=5)
subs_zero = substations[substations["p_mw"] == 0]
if len(subs_zero) > 0:
    subs_zero.plot(ax=ax, color=grey30, markersize=15, zorder=4, marker="x")
ax.set_title(f"UW load distribution (n={len(substations)})", fontsize=10)
ax.set_xlabel("Longitude [°E]", fontsize=8)
ax.set_ylabel("Latitude [°N]", fontsize=8)
ax.tick_params(labelsize=6)

# Right panel: load histogram
ax = axes[2]
substations[substations["p_mw"] > 0]["p_mw"].hist(ax=ax, bins=15, color=kitblue, edgecolor="white")
ax.set_title("UW load histogram", fontsize=10)
ax.set_xlabel("Last aktuell [MW]", fontsize=8)
ax.set_ylabel("Count", fontsize=8)
ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

# UW statistics per Gemeinde
gem_stats = substations.groupby("Gemeinde").agg(
    n_uw=("p_mw", "count"),
    total_load_mw=("p_mw", "sum"),
).reset_index()
print(f"\nGemeinden with UW: {len(gem_stats)}/{len(gemeinden)}")
print(f"Total regional load: {substations['p_mw'].sum():.1f} MW")

## 7. Save Outputs

- `source_regions.gpkg` -- 38 Gemeinden + population + GVA percent + area share
- `substations.gpkg` -- UW + load + Gemeinde assignment

In [ ]:
# Arrange output columns for source_regions
source_cols = [
    "geometry", "Name", "NUTS3",
    "population", "residential_percent",
    "agricultural_gva", "industrial_gva", "commercial_gva", "others_gva",
    "agricultural_percent", "industrial_percent", "commercial_percent", "others_percent",
    "area_m2", "area_percent",
]
source_regions = gemeinden[source_cols].copy()
source_regions = source_regions.reset_index(drop=True)

# Arrange output columns for substations
subs_cols = ["Kennzeichen", "Name", "p_mw", "Gemeinde", "NUTS3", "geometry"]
substations_out = substations[subs_cols].copy()
substations_out = substations_out.reset_index(drop=True)

# Save
source_regions.to_file(OUTPUT_DIR / "source_regions.gpkg", driver="GPKG")
substations_out.to_file(OUTPUT_DIR / "substations.gpkg", driver="GPKG")

print(f"Saved source_regions.gpkg: {len(source_regions)} rows")
print(f"Saved substations.gpkg: {len(substations_out)} rows")

print(f"\n=== source_regions overview ===")
print(source_regions.drop(columns="geometry").describe().T)

print(f"\n=== substations overview ===")
print(substations_out.drop(columns="geometry").describe())